МЕТОД ВЫЯВЛЕНИЯ И ОЦЕНКИ НИЗКО-СВЯЗАННЫХ КВАРТАЛОВ, ИМЕЮЩИХ ХАРАКТЕРИСТИКИ ЦЕНТРАЛЬНЫХ МЕСТ И ПОДВЕРЖЕННЫХ ВЛИЯНИЮ МАЯТНИКОВОЙ ТРУДОВОЙ МИГРАЦИИ

In [ ]:
import geopandas as gpd

blocks = gpd.read_file('blocks.geojson')
blocks['site_area'] = blocks.geometry.area


In [ ]:
buildings = gpd.read_file('buildings.geojson')
buildings_p = buildings.copy()
buildings_p.geometry = buildings_p.representative_point()
blocks = blocks.merge(
    blocks.sjoin(buildings_p, how='left', predicate='contains')
    .groupby('id')
    .agg({'population': 'sum'})
    .reset_index(),
    left_on='id',
    right_on='id'
)

In [ ]:
from blocksnet import Connectivity,AccessibilityProcessor

blocks = blocks.to_crs(epsg=32636)
ap = AccessibilityProcessor(blocks)
graph = ap.get_intermodal_graph()
acc_mx = ap.get_accessibility_matrix(graph)  

In [4]:
from blocksnet.models import City

city = City(
  blocks=blocks, 
  acc_mx=acc_mx, 
) 

In [5]:
from blocksnet import Connectivity

connectivity = Connectivity(city_model=city)
connect = connectivity.calculate()

In [ ]:
import matplotlib.pyplot as plt

PLOT_KWARGS = {"column": 'connectivity', "cmap": "plasma", "legend": True}
ax = connect.plot(color="#ddd", linewidth=0.1, figsize=(10, 10))
connect.plot(ax=ax, linewidth=0.1, **PLOT_KWARGS)
ax.axis('off')
plt.savefig('Связанность.png', dpi=900, bbox_inches='tight')

In [7]:
matrix = acc_mx.copy()

In [ ]:
import numpy as np

def min_max_normalization(data, new_min=0, new_max=1):
    min_value = np.min(data)
    max_value = np.max(data)
    normalized_data = (data - min_value) / (max_value - min_value) * (new_max - new_min) + new_min
    return normalized_data


# 
q1_landuse = {
    'industrial': 0.25,
    'business': 0.3,
    'special': 0.1,
    'transport': 0.1,
    'residential': 0.1,
    'NaN': 0.06,
    'agriculture': 0.05,
    'recreation': 0.05
}

blocks['q1'] = blocks['land_use'].apply(lambda x: q1_landuse.get(x, 0))
blocks['q2'] = min_max_normalization(np.sqrt(blocks['site_area']))
blocks['q3'] = blocks['q1'] * blocks['q2']
q3 = blocks['q3'].sum()
blocks['mark'] = blocks['q3'] / q3

In [13]:
blocks['population'] = blocks['population'].astype(int)

In [ ]:
import pandas as pd

c = blocks['population'].sum()
rng = np.random.default_rng(seed=0)
r = pd.Series(0, blocks.index)

blocks['mark'] = pd.to_numeric(blocks['mark'], errors='coerce')
blocks['mark'] = blocks['mark'].fillna(0)

p = blocks['mark'].values
p_sum = p.sum()

if p_sum == 0:
    p = np.ones_like(p) / len(p)
else:
    p = p / p_sum 

choice = np.unique(rng.choice(blocks.index, int(c), p=p), return_counts=True)
choice = r.add(pd.Series(choice[1], choice[0]), fill_value=0)


In [16]:
blocks['jobs'] = np.multiply(choice, 0.8).astype(int)

In [17]:
blocks['workers'] = blocks['population']

In [ ]:
blocks['jobs'].sum()

In [ ]:
blocks['population'].sum()

In [20]:
destination_matrix = pd.DataFrame(
    0,
    index=matrix.columns,
    columns=matrix.index,
)
blocks['jobs_left'] = blocks['jobs']
blocks['workers_left'] = blocks['workers']

In [ ]:
from pandarallel import pandarallel

pandarallel.initialize(progress_bar=False, verbose=0)


def workflows_loop_gravity(
        blocks: gpd.GeoDataFrame,
        distance_matrix: pd.DataFrame,
        selection_range,
        destination_matrix: pd.DataFrame,
):
    def calculate_visit_prob(x):
        import numpy as np
        k = 0.2

        if 0 <= x < 30:
            y = (np.exp(k * x) - 1) / (np.exp(30 * k) - 1) * 0.27 + 0.08
        elif 30 <= x <= 50:
            y = 0.60 * np.exp(-((x - 40) ** 2) / (2 * 9.63 ** 2))
        elif 50 < x:
            y = 0.35 * np.exp(-k * (x - 50))
        else:
            y = None  # Для значений x вне указанных диапазонов

        return y

    def apply_function_based_on_size(df, func, axis, threshold=500):
        if len(df) > threshold:
            return df.parallel_apply(func, axis=axis)
        else:
            return df.apply(func, axis=axis)

    def calculate_flows_workers(loc):
        import numpy as np
        import pandas as pd

        c = blocks.loc[loc.name]["jobs_left"]
        p = loc.apply(calculate_visit_prob)
        # threshold = p.quantile(best_choice)
        thres = p[p >= 0.15]
        if len(thres) > 0:
            p = thres
        p = p / p.sum()
        if p.sum() == 0:
            return loc
        rng = np.random.default_rng(seed=0)
        r = pd.Series(0, p.index)
        choice = np.unique(rng.choice(p.index, int(c), p=p.values), return_counts=True)
        choice = r.add(pd.Series(choice[1], choice[0]), fill_value=0)

        return choice

    def balance_flows_to_jobs(loc):
        import numpy as np
        import pandas as pd

        d = blocks.loc[loc.name]["workers_left"]
        loc = loc[loc > 0]
        if loc.sum() > 0:
            p = loc / loc.sum()
            rng = np.random.default_rng(seed=0)
            r = pd.Series(0, p.index)
            choice = np.unique(rng.choice(p.index, int(d), p=p.values), return_counts=True)
            choice = r.add(pd.Series(choice[1], choice[0]), fill_value=0)
            choice = pd.Series(
                data=np.minimum(loc.sort_index().values, choice.sort_index().values),
                index=loc.sort_index().index,
            )
            return choice
        return loc

    temp_destination_matrix = apply_function_based_on_size(
        distance_matrix, lambda x: calculate_flows_workers(x[x <= selection_range]), 1
    )

    temp_destination_matrix = temp_destination_matrix.fillna(0)

    temp_destination_matrix = apply_function_based_on_size(temp_destination_matrix, balance_flows_to_jobs, 0)

    temp_destination_matrix = temp_destination_matrix.fillna(0)
    destination_matrix = destination_matrix.add(temp_destination_matrix, fill_value=0)

    axis_1 = destination_matrix.sum(axis=1)
    axis_0 = destination_matrix.sum(axis=0)

    blocks["jobs_left"] = blocks["jobs"].subtract(axis_1, fill_value=0)
    blocks["workers_left"] = blocks["workers"].subtract(axis_0, fill_value=0)

    distance_matrix = distance_matrix.drop(
        index=blocks[blocks["jobs_left"] == 0].index.values,
        columns=blocks[blocks["workers_left"] == 0].index.values,
        errors="ignore",
    )

    selection_range = selection_range * 1.25
    print(selection_range)

    print(len(distance_matrix.columns), len(distance_matrix.index))

    if len(distance_matrix.columns) > 0 and len(distance_matrix.index) > 0:
        return workflows_loop_gravity(
            blocks, distance_matrix, selection_range, destination_matrix
        )
    return destination_matrix


res = workflows_loop_gravity(blocks, matrix.copy(), 60, destination_matrix)

In [22]:
def additional_options(
        blocks,
        matrix,
        destination_matrix,
):
    blocks["avg_dist"] = 0
    for i in range(len(destination_matrix)):
        loc = destination_matrix.iloc[i]
        distances_all = matrix.loc[loc.name]
        blocks["avg_dist"] = (
            blocks["avg_dist"]
            .add(distances_all.multiply(loc, fill_value=0), fill_value=0)

        )

    blocks["avg_dist"] = (blocks["avg_dist"] / (blocks["workers"] - blocks["workers_left"]))

    blocks["avg_dist"] = blocks.apply(
        lambda x: np.nan if (x["workers"] == x["workers_left"]) else round(x["avg_dist"], 2), axis=1
    )


additional_options(blocks, matrix, res)

In [23]:
blocks['jobs_applyed'] = blocks['jobs'] - blocks['jobs_left']
blocks['load%'] = blocks['jobs_applyed'] / blocks['jobs'] * 100

blocks['workers_non_applyed_%'] = (blocks['workers_left'] / blocks['workers']) * 100


In [ ]:
import matplotlib.pyplot as plt

PLOT_KWARGS = {"column": 'workers_non_applyed_%', "cmap": "RdYlGn_r", "vmin": 0, "vmax": 100, "legend": True}
ax = blocks.plot(color="#ddd", linewidth=0.1, figsize=(10, 10))
blocks.plot(ax=ax, linewidth=0.1, **PLOT_KWARGS)
ax.axis('off')
plt.savefig('20% дефицит нехватка мест время.png', dpi=900, bbox_inches='tight')

In [ ]:
m1 = blocks.reset_index().explore(column='workers_non_applyed_%', tiles='CartoDB positron')
m1

In [28]:
city = City(blocks = blocks,acc_mx=matrix)

In [29]:
from blocksnet import PopulationCentrality

centrality = PopulationCentrality(city_model=city)
result = centrality.calculate()

In [ ]:
result.explore(column='population_centrality',cmap='plasma')

In [ ]:
import matplotlib.pyplot as plt

PLOT_KWARGS = {"column": 'population_centrality', "cmap": "plasma", "vmin": 0, "vmax": 10, "legend": True}
ax = result.plot(color="#ddd", linewidth=0.1, figsize=(10, 10))
result.plot(ax=ax, linewidth=0.1, **PLOT_KWARGS)
ax.axis('off')
plt.savefig('Центральность населения.png', dpi=900, bbox_inches='tight')

In [34]:
blocks_new = pd.merge(blocks, result[['population_centrality']], left_index=True, right_index=True, how='inner')

In [35]:
blocks_new = pd.merge(blocks_new, connect[['connectivity']], left_index=True, right_index=True, how='inner')

In [ ]:
blocks_new['suffering_blocks'] = min_max_normalization(blocks_new['workers_non_applyed_%'] * blocks_new['population_centrality'] * blocks_new['connectivity'], 0, 10)
blocks_new.explore(column='suffering_blocks',cmap='plasma')

In [ ]:
import matplotlib.pyplot as plt

PLOT_KWARGS = {"column": 'suffering_blocks', "cmap": "plasma", "vmin": 0, "vmax": 10, "legend": True}
ax = blocks_new.plot(color="#ddd", linewidth=0.1, figsize=(10, 10))
blocks_new.plot(ax=ax, linewidth=0.1, **PLOT_KWARGS)
ax.axis('off')
plt.savefig('Страдающие кварталы.png', dpi=900, bbox_inches='tight')